# DeepVoice DACON — Colab 단계형 학습·평가·제출 워크플로우

ASVspoof 2019 LA/PA에서 시작해 Log-Mel baseline → AASIST → RawBoost → XLS-R → dual-graph → Mamba → 음성/음악 멀티브랜치 → 검증 기반 ensemble/calibration → DACON 코드 제출 패키지까지 연결하는 실험 노트북입니다.

이 노트북은 **한 번에 모든 모델을 학습하는 문서가 아닙니다.** `SELECTED_EXPERIMENT`로 한 실험씩 실행하고, 결과·체크포인트·검증 예측을 Google Drive에 누적합니다. 이렇게 해야 Colab 런타임 중단을 피하고 각 변경의 효과를 비교할 수 있습니다.

핵심 라벨 규약(ASVspoof): `0 = spoof/fake`, `1 = bonafide/real`. 모든 내부 score는 `P(bonafide)`로 저장합니다.


## 먼저 읽기 — 2026-08-20 기준 대회 상태와 규칙

- 대회: [딥보이스 범죄 대응을 위한 AI 탐지 모델 경진대회](https://dacon.io/competitions/official/236749/overview/description)
- 일정: 2026-08-26 시작, 2026-09-29 리더보드 제출 마감, 2026-09-30 종료.
- 오늘 기준 세부 데이터 설명, 평가 산식, 서버 사양, ZIP 용량·설치·추론 시간 제한은 **8월 26일 10:00 공개 예정**입니다.
- 제출 형식은 `submit.zip/{model/, script.py, requirements.txt}`이며 평가 서버는 인터넷이 차단됩니다.
- 비공개 TEST는 **추론 전용**입니다. 추가 학습, 튜닝, pseudo-labeling, TEST 전체 통계를 이용한 보정은 금지됩니다.
- 각 TEST 파일은 독립적으로 예측해야 합니다. 한 파일 내부 segment TTA는 허용되지만 다른 TEST 파일의 정보·예측·통계를 이용하면 안 됩니다.
- ASVspoof 등 외부 데이터와 공개 사전학습 모델은 현재 규칙상 사용할 수 있으나, 2차 평가 자료에 출처와 학습 데이터 전체를 명시·제출해야 합니다.

따라서 마지막 DACON 어댑터는 공개된 `sample_submission.csv`를 읽어 열을 검사하며, 알 수 없는 다중 출력 의미를 임의로 가정하지 않습니다. 공개일에 `DACON_SCHEMA`와 평가 산식만 확정하면 앞 단계의 학습 코드를 그대로 재사용할 수 있습니다.


## 0. Colab 런타임

권장: GPU 런타임. AASIST는 T4에서 batch 8, XLS-R 300M 계열은 batch 1–2와 gradient accumulation을 권장합니다. 아래 설치 셀 실행 후 import 오류가 남으면 런타임을 한 번 재시작합니다.


In [ ]:
%pip install -q "transformers>=4.48,<5" "accelerate>=1.2" "optuna>=4.0" \
  "soundfile>=0.12" "librosa>=0.10" "scikit-learn>=1.4" \
  "scipy>=1.11" "seaborn>=0.13" "pandas>=2.0"


In [ ]:
from __future__ import annotations

import gc
import hashlib
import importlib
import json
import math
import os
import random
import shutil
import subprocess
import sys
import tarfile
import time
import warnings
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from types import SimpleNamespace
from typing import Iterable, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from scipy.optimize import minimize
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, log_loss, roc_auc_score, roc_curve,
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


@dataclass
class ProjectConfig:
    drive_root: str = "/content/drive/MyDrive/deepvoice_dacon"
    work_root: str = "/content/deepvoice_work"
    # ZIP 이름은 Drive에 올린 실제 파일명으로 수정합니다.
    asv_archives: tuple[str, ...] = (
        "ASVspoof2019_LA.zip",
        "ASVspoof2019_PA.zip",
    )
    dacon_archive: str = "dacon_data.zip"
    sample_rate: int = 16_000
    clip_samples: int = 64_600
    seed: int = 42
    num_workers: int = 0  # Drive I/O 문제를 피하는 안전한 시작값


CFG = ProjectConfig()
DRIVE_ROOT = Path(CFG.drive_root)
WORK_ROOT = Path(CFG.work_root)
DATA_ROOT = WORK_ROOT / "data"
REPO_ROOT = WORK_ROOT / "repos"
RUN_ROOT = DRIVE_ROOT / "runs"
EXPORT_ROOT = DRIVE_ROOT / "exports"
for directory in (DRIVE_ROOT, WORK_ROOT, DATA_ROOT, REPO_ROOT, RUN_ROOT, EXPORT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(CFG.seed)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("torch/cuda:", torch.__version__, torch.version.cuda)


## 1. Google Drive ZIP 해제

Drive에서 매 epoch마다 오디오를 읽으면 느립니다. ZIP 원본과 체크포인트는 Drive에 보존하고, 학습 데이터는 Colab 로컬 `/content/deepvoice_work/data`로 한 번만 풉니다. 압축 해제 완료 marker가 있으면 재실행 시 건너뜁니다.


In [ ]:
def extract_archive(archive: Path, destination: Path, force: bool = False) -> Path:
    if not archive.exists():
        print(f"[skip] archive not found: {archive}")
        return destination
    digest = hashlib.sha1(str(archive.resolve()).encode()).hexdigest()[:10]
    marker = destination / f".extracted_{archive.stem}_{digest}"
    if marker.exists() and not force:
        print(f"[cached] {archive.name}")
        return destination

    destination.mkdir(parents=True, exist_ok=True)
    print(f"[extract] {archive} -> {destination}")
    if zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(destination)
    elif tarfile.is_tarfile(archive):
        with tarfile.open(archive) as tf:
            tf.extractall(destination)
    else:
        raise ValueError(f"지원하지 않는 압축 형식: {archive}")
    marker.write_text("ok\n", encoding="utf-8")
    return destination


EXTRACT_ASVSPOOF = True
EXTRACT_DACON = True  # 공개 전 파일이 없으면 안전하게 skip

if EXTRACT_ASVSPOOF:
    for name in CFG.asv_archives:
        extract_archive(DRIVE_ROOT / name, DATA_ROOT / "asvspoof2019")
if EXTRACT_DACON:
    extract_archive(DRIVE_ROOT / CFG.dacon_archive, DATA_ROOT / "dacon")


## 2. 공식 구현 준비 및 출처 고정

- AASIST: `clovaai/aasist` (MIT)
- RawBoost: `TakHemlata/RawBoost-antispoofing`
- SSL anti-spoofing: `TakHemlata/SSL_Anti-spoofing` (MIT, 선택 실험)
- XLSR-Mamba: `swagshaw/XLSR-Mamba` (MIT, 선택 실험)

핵심 노트북은 현재 Colab의 PyTorch를 유지합니다. 오래된 fairseq 버전을 요구하는 SSL/Mamba 공식 재현은 별도 런타임에서 실행하는 것이 안전하므로 기본 clone 대상에서 제외했습니다. 사용한 저장소 commit은 자동 기록됩니다.


In [ ]:
REPOSITORIES = {
    "aasist": "https://github.com/clovaai/aasist.git",
    "rawboost": "https://github.com/TakHemlata/RawBoost-antispoofing.git",
    "ssl_antispoof": "https://github.com/TakHemlata/SSL_Anti-spoofing.git",
    "xlsr_mamba": "https://github.com/swagshaw/XLSR-Mamba.git",
}


def clone_repo(name: str, optional: bool = False) -> Path:
    destination = REPO_ROOT / name
    if not destination.exists():
        print("cloning", REPOSITORIES[name])
        subprocess.run(
            ["git", "clone", "--depth", "1", REPOSITORIES[name], str(destination)],
            check=True,
        )
    commit = subprocess.check_output(
        ["git", "-C", str(destination), "rev-parse", "HEAD"], text=True
    ).strip()
    print(f"{name}: {commit}")
    return destination


CLONE_CORE_REPOS = True
CLONE_LEGACY_ADVANCED_REPOS = False

REPO_COMMITS = {}
if CLONE_CORE_REPOS:
    for repo_name in ("aasist", "rawboost"):
        repo_path = clone_repo(repo_name)
        REPO_COMMITS[repo_name] = subprocess.check_output(
            ["git", "-C", str(repo_path), "rev-parse", "HEAD"], text=True
        ).strip()
if CLONE_LEGACY_ADVANCED_REPOS:
    for repo_name in ("ssl_antispoof", "xlsr_mamba"):
        repo_path = clone_repo(repo_name, optional=True)
        REPO_COMMITS[repo_name] = subprocess.check_output(
            ["git", "-C", str(repo_path), "rev-parse", "HEAD"], text=True
        ).strip()


## 3. ASVspoof 2019 LA/PA 인덱스 생성

공식 protocol의 train/dev/eval split을 그대로 사용합니다. Random split을 다시 만들지 않습니다. LA와 PA를 합칠 때는 `track × label` 균형 sampler를 사용해 PA 또는 spoof 다수가 학습을 지배하지 않게 합니다.


In [ ]:
AUDIO_SUFFIXES = {".wav", ".flac", ".ogg", ".mp3", ".m4a", ".aac"}


def find_protocol(root: Path, track: str, split: str) -> Path | None:
    candidates = []
    for path in root.rglob("*.txt"):
        name = path.name.lower()
        full = str(path).lower()
        if track.lower() in full and split.lower() in name and "protocol" in full:
            candidates.append(path)
    if not candidates:
        return None
    candidates.sort(key=lambda p: ("cm." not in p.name.lower(), len(str(p))))
    return candidates[0]


def build_audio_map(root: Path) -> dict[str, Path]:
    mapping = {}
    for path in tqdm(root.rglob("*"), desc="Audio index"):
        if path.is_file() and path.suffix.lower() in AUDIO_SUFFIXES:
            mapping.setdefault(path.stem, path)
    return mapping


def parse_asvspoof_protocol(protocol: Path, audio_map: dict[str, Path], track: str, split: str) -> pd.DataFrame:
    rows = []
    for line_no, line in enumerate(protocol.read_text(encoding="utf-8").splitlines(), 1):
        parts = line.split()
        if len(parts) < 5:
            raise ValueError(f"잘못된 protocol line {protocol}:{line_no}: {line}")
        speaker, utt_id, environment, attack, key = parts[:5]
        key = key.lower()
        if key not in {"bonafide", "spoof"}:
            raise ValueError(f"알 수 없는 label {key}: {protocol}:{line_no}")
        rows.append({
            "id": utt_id,
            "path": str(audio_map.get(utt_id, "")),
            "label": 1 if key == "bonafide" else 0,
            "key": key,
            "speaker": speaker,
            "environment": environment,
            "attack": attack,
            "track": track,
            "split": split,
            "source": "ASVspoof2019",
        })
    frame = pd.DataFrame(rows)
    missing = frame[frame["path"] == ""]
    if len(missing):
        examples = missing["id"].head().tolist()
        raise FileNotFoundError(f"protocol 오디오 {len(missing)}개를 찾지 못함. 예: {examples}")
    return frame


def build_asvspoof_index(root: Path) -> pd.DataFrame:
    if not root.exists():
        print("ASVspoof root가 없습니다:", root)
        return pd.DataFrame()
    audio_map = build_audio_map(root)
    frames = []
    for track in ("LA", "PA"):
        for split in ("train", "dev", "eval"):
            protocol = find_protocol(root, track, split)
            if protocol is None:
                print(f"[warn] protocol 없음: {track}/{split}")
                continue
            print(f"{track}/{split}: {protocol}")
            frames.append(parse_asvspoof_protocol(protocol, audio_map, track, split))
    if not frames:
        return pd.DataFrame()
    frame = pd.concat(frames, ignore_index=True)
    assert not frame.duplicated(["track", "split", "id"]).any()
    return frame


ASV_ROOT = DATA_ROOT / "asvspoof2019"
asv_df = build_asvspoof_index(ASV_ROOT)
if len(asv_df):
    display(pd.crosstab([asv_df["track"], asv_df["split"]], asv_df["key"], margins=True))
    display(asv_df.head())


## 4. DACON 공개 데이터 계약 검사

8월 26일 이후 실행합니다. 파일명·열 이름을 추측해 학습하는 대신 실제 CSV와 오디오를 먼저 출력하고 검증합니다. `test`에는 label이 없어야 하며, 이후 학습 파이프라인에는 오직 공개 train과 외부 학습 데이터만 넣습니다.


In [ ]:
DACON_ROOT = DATA_ROOT / "dacon"


def csv_inventory(root: Path) -> dict[str, Path]:
    if not root.exists():
        return {}
    return {p.name.lower(): p for p in root.rglob("*.csv")}


def audio_inventory(root: Path) -> pd.DataFrame:
    if not root.exists():
        return pd.DataFrame(columns=["id", "path"])
    paths = [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_SUFFIXES]
    return pd.DataFrame({"id": [p.stem for p in paths], "path": [str(p) for p in paths]})


def inspect_dacon_contract(root: Path) -> dict:
    csvs = csv_inventory(root)
    print("CSV files:", {k: str(v) for k, v in csvs.items()})
    result = {"csvs": csvs, "train": None, "test": None, "sample": None}
    for key, path in csvs.items():
        frame = pd.read_csv(path)
        print(f"\n[{path.name}] shape={frame.shape}")
        display(frame.head())
        print(frame.dtypes)
        if "sample" in key and "submission" in key:
            result["sample"] = frame
            result["sample_path"] = path
        elif "train" in key:
            result["train"] = frame
            result["train_path"] = path
        elif "test" in key:
            result["test"] = frame
            result["test_path"] = path
    audio_df = audio_inventory(root)
    print("audio files:", len(audio_df), "suffixes:", sorted({Path(x).suffix for x in audio_df["path"]}))
    result["audio"] = audio_df
    return result


dacon_contract = inspect_dacon_contract(DACON_ROOT)

# 공개 후 반드시 명시적으로 확인/수정할 값. None이면 sample_submission에서 보수적으로 추론합니다.
DACON_SCHEMA = {
    "id_col": None,
    "audio_col": None,
    "target_cols": None,
    "group_col": None,  # speaker/source/generator 열이 있으면 지정
    # 단일 이진 출력일 때만 사용: "spoof" 또는 "bonafide"
    "single_target_positive": "spoof",
}


def infer_schema(contract: dict, overrides: dict) -> dict:
    sample = contract.get("sample")
    test = contract.get("test")
    if sample is None:
        raise FileNotFoundError("sample_submission.csv가 없습니다. 8월 26일 공개 데이터를 확인하세요.")
    id_col = overrides.get("id_col") or sample.columns[0]
    target_cols = overrides.get("target_cols") or [c for c in sample.columns if c != id_col]
    audio_col = overrides.get("audio_col")
    if audio_col is None and test is not None:
        candidates = [c for c in test.columns if any(k in c.lower() for k in ("path", "file", "audio", "name"))]
        audio_col = candidates[0] if candidates else None
    return {**overrides, "id_col": id_col, "target_cols": target_cols, "audio_col": audio_col}


def prepare_dacon_supervised_frames(contract: dict, schema: dict, dev_size: float = 0.2):
    """공개 train만 사용해 train/dev를 만든다. TEST는 이 함수 입력에 들어오지 않는다."""
    train = contract.get("train")
    if train is None:
        raise FileNotFoundError("공개 train.csv가 없습니다.")
    train = train.copy()
    id_col = schema["id_col"]
    target_cols = list(schema["target_cols"])
    missing_targets = [c for c in target_cols if c not in train.columns]
    if missing_targets:
        raise KeyError(f"train.csv에 target 열이 없습니다: {missing_targets}")

    audio = contract["audio"].copy()
    audio["audio_key"] = audio["id"].astype(str).map(lambda x: Path(x).stem)
    if audio["audio_key"].duplicated().any():
        raise ValueError("오디오 stem이 중복됩니다. train/test 하위 폴더를 포함하도록 path resolver를 수정하세요.")
    audio_map = dict(zip(audio["audio_key"], audio["path"]))
    ref_col = schema.get("audio_col") if schema.get("audio_col") in train.columns else id_col
    train["id"] = train[id_col].astype(str)
    train["audio_key"] = train[ref_col].astype(str).map(lambda x: Path(x).stem)
    train["path"] = train["audio_key"].map(audio_map)
    if train["path"].isna().any():
        raise FileNotFoundError(f"train 오디오 매칭 실패 {int(train['path'].isna().sum())}개")

    if len(target_cols) == 1:
        values = train[target_cols[0]].astype(int)
        if not set(values.unique()).issubset({0, 1}):
            raise ValueError("단일 target은 0/1이어야 합니다.")
        train["label"] = 1 - values if schema["single_target_positive"] == "spoof" else values
        label_cols: str | list[str] = "label"  # 내부 규약: 1=bonafide
    else:
        label_cols = target_cols

    group_col = schema.get("group_col")
    indices = np.arange(len(train))
    if group_col and group_col in train.columns:
        splitter = GroupShuffleSplit(n_splits=1, test_size=dev_size, random_state=CFG.seed)
        train_idx, dev_idx = next(splitter.split(indices, groups=train[group_col]))
    else:
        stratify = train[target_cols[0]] if len(target_cols) == 1 else None
        train_idx, dev_idx = train_test_split(
            indices, test_size=dev_size, random_state=CFG.seed, stratify=stratify,
        )
    train_part = train.iloc[train_idx].reset_index(drop=True)
    dev_part = train.iloc[dev_idx].reset_index(drop=True)
    assert set(train_part["id"]).isdisjoint(set(dev_part["id"]))
    return train_part, dev_part, target_cols, label_cols


## 5. EDA — 길이·샘플레이트·채널·RMS·peak·ZCR

전체 파일을 매번 읽지 않도록 먼저 최대 2,000개를 표본 조사합니다. 모델 선택 전에 sample rate 혼합, 무음/클리핑, 길이 꼬리, 클래스/track/attack 분포를 확인합니다.


In [ ]:
def audio_metadata(path: str) -> dict:
    info = sf.info(path)
    audio, sr = sf.read(path, dtype="float32", always_2d=True)
    mono = audio.mean(axis=1)
    rms = float(np.sqrt(np.mean(np.square(mono)) + 1e-12))
    peak = float(np.max(np.abs(mono))) if len(mono) else 0.0
    zcr = float(np.mean(mono[:-1] * mono[1:] < 0)) if len(mono) > 1 else 0.0
    return {
        "duration": info.frames / info.samplerate,
        "sample_rate": info.samplerate,
        "channels": info.channels,
        "rms": rms,
        "peak": peak,
        "zcr": zcr,
    }


def run_eda(frame: pd.DataFrame, max_files: int = 2000) -> pd.DataFrame:
    if frame.empty:
        print("EDA 대상 데이터가 없습니다.")
        return pd.DataFrame()
    sample = frame.sample(min(max_files, len(frame)), random_state=CFG.seed).copy()
    metadata = []
    errors = []
    for row in tqdm(sample.itertuples(index=False), total=len(sample), desc="EDA metadata"):
        try:
            metadata.append({"id": row.id, **audio_metadata(row.path)})
        except Exception as exc:
            errors.append((row.path, repr(exc)))
    meta = pd.DataFrame(metadata)
    result = sample.merge(meta, on="id", how="left")
    if errors:
        print("read errors:", errors[:5])
    display(result.describe(include="all").T)
    return result


eda_df = run_eda(asv_df[asv_df["split"] == "train"] if len(asv_df) else pd.DataFrame())


In [ ]:
if len(eda_df):
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    sns.histplot(data=eda_df, x="duration", hue="track", bins=50, ax=axes[0, 0])
    sns.countplot(data=eda_df, x="sample_rate", ax=axes[0, 1])
    sns.countplot(data=eda_df, x="channels", ax=axes[0, 2])
    sns.histplot(data=eda_df, x="rms", hue="key", bins=50, ax=axes[1, 0])
    sns.histplot(data=eda_df, x="peak", hue="key", bins=50, ax=axes[1, 1])
    sns.histplot(data=eda_df, x="zcr", hue="key", bins=50, ax=axes[1, 2])
    plt.tight_layout()
    plt.show()


def show_examples(frame: pd.DataFrame, n: int = 4) -> None:
    if frame.empty:
        return
    picks = frame.groupby(["track", "key"], group_keys=False).head(1).head(n)
    fig, axes = plt.subplots(len(picks), 2, figsize=(14, 3 * len(picks)), squeeze=False)
    for row_idx, row in enumerate(picks.itertuples(index=False)):
        wav, sr = torchaudio.load(row.path)
        wav = wav.mean(0)
        if sr != CFG.sample_rate:
            wav = torchaudio.functional.resample(wav, sr, CFG.sample_rate)
        axes[row_idx, 0].plot(wav[: CFG.sample_rate * 6].numpy())
        axes[row_idx, 0].set_title(f"{row.track}/{row.key}/{row.id}")
        spec = torch.stft(wav[: CFG.sample_rate * 6], n_fft=512, hop_length=160, return_complex=True).abs()
        axes[row_idx, 1].imshow(torch.log1p(spec).numpy(), origin="lower", aspect="auto")
    plt.tight_layout()
    plt.show()


show_examples(asv_df[asv_df["split"] == "train"] if len(asv_df) else pd.DataFrame())


## 6. RawBoost·통신환경 augmentation

`OfficialRawBoost`는 공식 `RawBoost.py`의 함수를 직접 사용합니다. 학습 데이터에만 확률적으로 적용됩니다. 8 kHz 왕복 resampling·gain·clipping은 별도 통신환경 보강이며, validation/TEST에는 절대 랜덤 augmentation을 적용하지 않습니다.


In [ ]:
def import_rawboost_module():
    path = REPO_ROOT / "rawboost" / "RawBoost.py"
    if not path.exists():
        raise FileNotFoundError("RawBoost repo가 없습니다. 2절 clone 셀을 실행하세요.")
    spec = importlib.util.spec_from_file_location("official_rawboost", path)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


class OfficialRawBoost:
    """공식 RawBoost 함수 기반. algo 5=(LnL + ISD), algo 4=(LnL + ISD + SSI)."""

    def __init__(self, probability: float = 0.5, algo: int = 5, sample_rate: int = 16000):
        self.p = probability
        self.algo = algo
        self.sr = sample_rate
        self.rb = import_rawboost_module()

    def _lnl(self, x: np.ndarray) -> np.ndarray:
        return self.rb.LnL_convolutive_noise(
            x, N_f=5, nBands=5, minF=20, maxF=min(8000, self.sr // 2 - 1),
            minBW=100, maxBW=1000, minCoeff=10, maxCoeff=100,
            minG=0, maxG=0, minBiasLinNonLin=5, maxBiasLinNonLin=20, fs=self.sr,
        )

    def _isd(self, x: np.ndarray) -> np.ndarray:
        return self.rb.ISD_additive_noise(x, P=10, g_sd=2)

    def _ssi(self, x: np.ndarray) -> np.ndarray:
        return self.rb.SSI_additive_noise(
            x, SNRmin=10, SNRmax=40, nBands=5, minF=20,
            maxF=min(8000, self.sr // 2 - 1), minBW=100, maxBW=1000,
            minCoeff=10, maxCoeff=100, minG=0, maxG=0, fs=self.sr,
        )

    def __call__(self, waveform: torch.Tensor) -> torch.Tensor:
        if random.random() >= self.p:
            return waveform
        x = waveform.detach().cpu().numpy().astype(np.float64)
        if self.algo in (1, 4, 5, 6):
            x = self._lnl(x)
        if self.algo in (2, 4, 5, 7):
            x = self._isd(x)
        if self.algo in (3, 4, 6, 7):
            x = self._ssi(x)
        if self.algo == 8:
            x = self._lnl(x) + self._isd(x)
            x = self.rb.normWav(x, 0)
        return torch.from_numpy(np.asarray(x, dtype=np.float32))


class CommunicationAugment:
    def __init__(self, probability: float = 0.35, sample_rate: int = 16000):
        self.p = probability
        self.sr = sample_rate

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        if random.random() >= self.p:
            return x
        choice = random.choice(("telephone", "gain", "clip", "noise"))
        if choice == "telephone":
            x = torchaudio.functional.resample(x, self.sr, 8000)
            x = torchaudio.functional.resample(x, 8000, self.sr)
        elif choice == "gain":
            x = x * (10 ** (random.uniform(-8, 6) / 20))
        elif choice == "clip":
            limit = random.uniform(0.3, 0.9)
            x = torch.clamp(x, -limit, limit) / limit
        else:
            signal_power = x.square().mean().clamp_min(1e-8)
            snr_db = random.uniform(15, 35)
            noise_power = signal_power / (10 ** (snr_db / 10))
            x = x + torch.randn_like(x) * noise_power.sqrt()
        return x.clamp(-1, 1)


## 7. 공통 Dataset / DataLoader

짧은 파일은 AASIST 공식 방식처럼 반복 padding하고, 긴 파일은 train에서 random crop, dev/test에서 deterministic center crop을 씁니다. 제출 추론은 뒤에서 한 파일의 여러 segment를 평균냅니다.


In [ ]:
def load_mono(path: str, target_sr: int = 16000) -> torch.Tensor:
    wav, sr = torchaudio.load(path)
    wav = wav.float().mean(0)
    if wav.numel() == 0:
        raise ValueError(f"빈 오디오: {path}")
    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)
    return wav


def fixed_length(wav: torch.Tensor, length: int, train: bool) -> torch.Tensor:
    if wav.numel() >= length:
        if train:
            start = random.randint(0, wav.numel() - length)
        else:
            start = (wav.numel() - length) // 2
        return wav[start : start + length]
    repeats = math.ceil(length / wav.numel())
    return wav.repeat(repeats)[:length]


class AudioFrameDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        label_cols: str | Sequence[str] = "label",
        train: bool = False,
        clip_samples: int = 64_600,
        rawboost_probability: float = 0.0,
        rawboost_algo: int = 5,
        communication_probability: float = 0.0,
    ):
        self.frame = frame.reset_index(drop=True).copy()
        self.label_cols = [label_cols] if isinstance(label_cols, str) else list(label_cols)
        self.train = train
        self.clip_samples = clip_samples
        self.rawboost = (
            OfficialRawBoost(rawboost_probability, rawboost_algo, CFG.sample_rate)
            if rawboost_probability > 0 else None
        )
        self.communication = CommunicationAugment(communication_probability, CFG.sample_rate)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> dict:
        row = self.frame.iloc[index]
        wav = load_mono(row["path"], CFG.sample_rate)
        if self.train and self.rawboost is not None:
            wav = self.rawboost(wav)
        if self.train:
            wav = self.communication(wav)
        wav = fixed_length(wav, self.clip_samples, self.train)
        labels = row[self.label_cols].to_numpy(dtype=np.float32)
        if len(self.label_cols) == 1:
            label = torch.tensor(int(labels[0]), dtype=torch.long)
        else:
            label = torch.tensor(labels, dtype=torch.float32)
        return {"audio": wav, "label": label, "id": str(row["id"])}


def balanced_sampler(frame: pd.DataFrame) -> WeightedRandomSampler:
    group_cols = [c for c in ("track", "label") if c in frame.columns]
    keys = frame[group_cols].astype(str).agg("|".join, axis=1)
    counts = keys.value_counts()
    weights = keys.map(lambda key: 1.0 / counts[key]).to_numpy()
    return WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), len(weights), replacement=True)


def make_loaders(train_frame: pd.DataFrame, dev_frame: pd.DataFrame, exp: dict):
    label_cols = exp.get("label_cols", "label")
    train_ds = AudioFrameDataset(
        train_frame,
        label_cols=label_cols,
        train=True,
        clip_samples=exp["clip_samples"],
        rawboost_probability=exp.get("rawboost_probability", 0.0),
        rawboost_algo=exp.get("rawboost_algo", 5),
        communication_probability=exp.get("communication_probability", 0.0),
    )
    dev_ds = AudioFrameDataset(
        dev_frame, label_cols=label_cols, train=False, clip_samples=exp["clip_samples"]
    )
    sampler = balanced_sampler(train_frame) if exp.get("balanced", True) else None
    train_loader = DataLoader(
        train_ds, batch_size=exp["batch_size"], sampler=sampler,
        shuffle=sampler is None, num_workers=CFG.num_workers, pin_memory=True,
        drop_last=True,
    )
    dev_loader = DataLoader(
        dev_ds, batch_size=exp.get("eval_batch_size", exp["batch_size"]),
        shuffle=False, num_workers=CFG.num_workers, pin_memory=True,
    )
    return train_loader, dev_loader


## 8. 공통 지표

ASVspoof binary validation에는 EER, AUC, accuracy, F1을 함께 기록합니다. EER score 방향은 `높을수록 bonafide`입니다. DACON 공식 산식이 공개되면 `selection_metric`을 해당 산식으로 바꿉니다.


In [ ]:
def compute_eer(y_true: np.ndarray, score_bonafide: np.ndarray) -> tuple[float, float]:
    fpr, tpr, thresholds = roc_curve(y_true, score_bonafide, pos_label=1)
    fnr = 1 - tpr
    idx = int(np.nanargmin(np.abs(fnr - fpr)))
    return float((fnr[idx] + fpr[idx]) / 2), float(thresholds[idx])


def binary_metrics(y_true: np.ndarray, score_bonafide: np.ndarray) -> dict:
    eer, eer_threshold = compute_eer(y_true, score_bonafide)
    pred = (score_bonafide >= 0.5).astype(int)
    return {
        "eer": eer,
        "eer_threshold": eer_threshold,
        "auc": roc_auc_score(y_true, score_bonafide),
        "accuracy": accuracy_score(y_true, pred),
        "f1": f1_score(y_true, pred),
    }


def multilabel_metrics(y_true: np.ndarray, probabilities: np.ndarray) -> dict:
    pred = (probabilities >= 0.5).astype(int)
    aucs = []
    for idx in range(y_true.shape[1]):
        if np.unique(y_true[:, idx]).size == 2:
            aucs.append(roc_auc_score(y_true[:, idx], probabilities[:, idx]))
    return {
        "macro_auc": float(np.mean(aucs)) if aucs else float("nan"),
        "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
    }


## 9. EXP01 — Log-Mel CNN sanity baseline

빠른 파이프라인 검증용입니다. 이 모델이 학습되지 않으면 AASIST로 넘어가지 않습니다.


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.SiLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.SiLU(),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.block(x)


class LogMelCNN(nn.Module):
    def __init__(self, n_mels: int = 96, dropout: float = 0.25, num_outputs: int = 2):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.sample_rate, n_fft=1024, win_length=400,
            hop_length=160, n_mels=n_mels, f_min=20, f_max=7600, power=2,
        )
        self.encoder = nn.Sequential(
            ConvBlock(1, 32), ConvBlock(32, 64), ConvBlock(64, 128),
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
        )
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(128, num_outputs))

    def features(self, audio: torch.Tensor) -> torch.Tensor:
        x = torch.log(self.mel(audio).clamp_min(1e-6))
        x = (x - x.mean(dim=(-2, -1), keepdim=True)) / (x.std(dim=(-2, -1), keepdim=True) + 1e-5)
        return self.encoder(x.unsqueeze(1))

    def forward(self, audio: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(audio))


## 10. EXP02–03 — 공식 AASIST / AASIST + RawBoost

AASIST 구조는 공식 저장소 코드를 그대로 import합니다. augmentation만 Dataset에서 교체합니다. `freq_aug`는 RawBoost와 별개의 모델 내부 옵션이며 한 번에 너무 많은 augmentation을 켜지 않도록 기본값은 `False`입니다.


In [ ]:
def load_official_aasist(freq_aug: bool = False) -> nn.Module:
    repo = REPO_ROOT / "aasist"
    config_path = repo / "config" / "AASIST.conf"
    if not config_path.exists():
        raise FileNotFoundError("AASIST repo/config가 없습니다. 2절 clone 셀을 실행하세요.")
    with config_path.open(encoding="utf-8") as file:
        model_config = json.load(file)["model_config"]
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    from models.AASIST import Model as AASISTModel

    class Wrapper(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = AASISTModel(model_config)
            self.freq_aug = freq_aug

        def forward(self, audio: torch.Tensor) -> torch.Tensor:
            _, logits = self.net(audio, Freq_aug=self.freq_aug and self.training)
            return logits

    return Wrapper()


## 11. EXP04–06 — XLS-R pooling / XLS-R + AASIST-inspired dual graph

`XLSRPoolClassifier`는 SSL 표현의 attentive pooling baseline입니다. `XLSRDualGraphClassifier`는 XLS-R frame 특징에 시간/특징 두 view의 graph-attention을 적용한 **AASIST-inspired** 구현이며, 공식 raw-waveform AASIST와 동일한 모델이라고 부르지 않습니다. 논문 재현이 필요한 경우 뒤의 공식 SSL_Anti-spoofing 별도 런타임 절차를 사용하세요.


In [ ]:
from transformers import AutoModel


class SSLBackbone(nn.Module):
    def __init__(self, model_name: str, freeze: bool = True):
        super().__init__()
        self.ssl = AutoModel.from_pretrained(model_name)
        self.hidden_size = self.ssl.config.hidden_size
        self.freeze_all() if freeze else None

    def freeze_all(self):
        for parameter in self.ssl.parameters():
            parameter.requires_grad = False

    def unfreeze_last_n(self, n: int = 4):
        self.freeze_all()
        encoder = getattr(self.ssl, "encoder", None)
        layers = getattr(encoder, "layers", None)
        if layers is None:
            raise AttributeError("이 backbone에서 encoder.layers를 찾지 못했습니다.")
        for layer in layers[-n:]:
            for parameter in layer.parameters():
                parameter.requires_grad = True

    def forward_features(self, audio: torch.Tensor) -> torch.Tensor:
        if not any(p.requires_grad for p in self.ssl.parameters()):
            with torch.no_grad():
                return self.ssl(audio).last_hidden_state
        return self.ssl(audio).last_hidden_state


class AttentivePool(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(dim, dim // 2), nn.Tanh(), nn.Linear(dim // 2, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        weights = torch.softmax(self.score(x), dim=1)
        mean = (x * weights).sum(dim=1)
        variance = (weights * (x - mean[:, None]).square()).sum(dim=1).clamp_min(1e-6)
        return torch.cat([mean, variance.sqrt()], dim=-1)


class XLSRPoolClassifier(SSLBackbone):
    def __init__(self, model_name: str, freeze: bool = True, dropout: float = 0.2, num_outputs: int = 2):
        super().__init__(model_name, freeze)
        self.pool = AttentivePool(self.hidden_size)
        self.head = nn.Sequential(
            nn.LayerNorm(self.hidden_size * 2), nn.Dropout(dropout),
            nn.Linear(self.hidden_size * 2, 256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, num_outputs),
        )

    def forward(self, audio: torch.Tensor) -> torch.Tensor:
        return self.head(self.pool(self.forward_features(audio)))


class GraphBlock(nn.Module):
    def __init__(self, dim: int, heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim * 4, dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.norm1(x)
        x = x + self.attn(z, z, z, need_weights=False)[0]
        return x + self.ff(self.norm2(x))


class XLSRDualGraphClassifier(SSLBackbone):
    def __init__(self, model_name: str, freeze: bool = True, dim: int = 128, dropout: float = 0.2, num_outputs: int = 2):
        super().__init__(model_name, freeze)
        self.proj = nn.Linear(self.hidden_size, dim)
        self.feature_node_proj = nn.Linear(8, dim)
        self.time_graph = nn.Sequential(GraphBlock(dim), GraphBlock(dim))
        self.feature_graph = nn.Sequential(GraphBlock(dim), GraphBlock(dim))
        self.head = nn.Sequential(
            nn.LayerNorm(dim * 4), nn.Dropout(dropout),
            nn.Linear(dim * 4, 256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, num_outputs),
        )

    def forward(self, audio: torch.Tensor) -> torch.Tensor:
        h = self.proj(self.forward_features(audio))            # [B,T,D]
        time_nodes = F.adaptive_avg_pool1d(h.transpose(1, 2), 64).transpose(1, 2)
        feature_nodes = F.adaptive_avg_pool1d(h.transpose(1, 2), 8)
        feature_nodes = self.feature_node_proj(feature_nodes)  # [B,D,D]
        t = self.time_graph(time_nodes)
        f = self.feature_graph(feature_nodes)
        pooled = torch.cat([t.mean(1), t.amax(1), f.mean(1), f.amax(1)], dim=-1)
        return self.head(pooled)


## 12. EXP07–08 — Bi-Mamba와 음성/음악 멀티브랜치

Mamba는 Colab CUDA/PyTorch 조합에 맞는 wheel 또는 빌드가 필요하므로 선택 설치입니다. 대회 label이 공개되기 전에는 멀티브랜치 출력 수와 의미를 확정할 수 없습니다. `num_outputs`를 `sample_submission`의 target 열 수와 맞춘 뒤 공개 train label로 학습합니다.


In [ ]:
# Mamba 실험을 실행할 런타임에서만 설치:
# %pip install -q "mamba-ssm>=2.2" causal-conv1d


class XLSRBiMambaClassifier(SSLBackbone):
    def __init__(self, model_name: str, freeze: bool = True, dim: int = 256, dropout: float = 0.2, num_outputs: int = 2):
        super().__init__(model_name, freeze)
        try:
            from mamba_ssm import Mamba
        except ImportError as exc:
            raise ImportError("위 선택 설치 셀로 mamba-ssm을 설치한 뒤 런타임을 재시작하세요.") from exc
        self.proj = nn.Linear(self.hidden_size, dim)
        self.forward_mamba = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        self.backward_mamba = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        self.pool = AttentivePool(dim * 2)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(dim * 4, num_outputs))

    def forward(self, audio: torch.Tensor) -> torch.Tensor:
        h = self.proj(self.forward_features(audio))
        forward = self.forward_mamba(h)
        backward = torch.flip(self.backward_mamba(torch.flip(h, dims=(1,))), dims=(1,))
        return self.head(self.pool(torch.cat([forward, backward], dim=-1)))


class SpeechMusicMultiBranch(SSLBackbone):
    def __init__(self, model_name: str, num_outputs: int, freeze: bool = True, dropout: float = 0.25):
        super().__init__(model_name, freeze)
        self.speech_pool = AttentivePool(self.hidden_size)
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.sample_rate, n_fft=1024, hop_length=160, n_mels=128,
        )
        self.music_encoder = nn.Sequential(
            ConvBlock(1, 32), ConvBlock(32, 64), ConvBlock(64, 128),
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(self.hidden_size * 2 + 128, 384), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(384, num_outputs),
        )

    def forward(self, audio: torch.Tensor) -> torch.Tensor:
        speech = self.speech_pool(self.forward_features(audio))
        mel = torch.log(self.mel(audio).clamp_min(1e-6))
        music = self.music_encoder(mel.unsqueeze(1))
        return self.fusion(torch.cat([speech, music], dim=-1))


## 13. 실험 registry — 검증을 위한 권장 시작값

“최적 파라미터”는 데이터와 평가 산식 없이 미리 존재하지 않습니다. 아래 값은 Colab T4에서 시작하기 좋은 안전한 값이고, 진짜 선택은 dev/OOF 점수와 Optuna 탐색으로 합니다. LA/PA는 별도 AASIST와 혼합 강건 모델을 모두 남기는 구성이 좋습니다.


In [ ]:
XLSR_MODEL = "facebook/wav2vec2-xls-r-300m"

EXPERIMENTS = {
    "exp001_logmel_lapa": dict(
        model="logmel", tracks=("LA", "PA"), batch_size=32, eval_batch_size=64,
        grad_accum=1, epochs=20, lr=3e-4, backbone_lr=3e-4, weight_decay=1e-4,
        clip_samples=64_600, dropout=0.25, balanced=True, patience=5,
    ),
    "exp002_aasist_la": dict(
        model="aasist", tracks=("LA",), batch_size=8, eval_batch_size=16,
        grad_accum=3, epochs=20, lr=1e-4, backbone_lr=1e-4, weight_decay=1e-4,
        clip_samples=64_600, freq_aug=False, balanced=True, patience=6,
    ),
    "exp002p_aasist_pa": dict(
        model="aasist", tracks=("PA",), batch_size=8, eval_batch_size=16,
        grad_accum=3, epochs=20, lr=1e-4, backbone_lr=1e-4, weight_decay=1e-4,
        clip_samples=64_600, freq_aug=False, balanced=True, patience=6,
    ),
    "exp003_aasist_rawboost_lapa": dict(
        model="aasist", tracks=("LA", "PA"), batch_size=8, eval_batch_size=16,
        grad_accum=3, epochs=24, lr=8e-5, backbone_lr=8e-5, weight_decay=1e-4,
        clip_samples=64_600, rawboost_probability=0.5, rawboost_algo=5,
        communication_probability=0.25, freq_aug=False, balanced=True, patience=7,
    ),
    "exp004_xlsr_pool_lapa": dict(
        model="xlsr_pool", tracks=("LA", "PA"), batch_size=2, eval_batch_size=4,
        grad_accum=8, epochs=10, lr=2e-4, backbone_lr=1e-6, weight_decay=1e-4,
        clip_samples=96_000, freeze=True, freeze_epochs=2, unfreeze_last_n=4,
        dropout=0.20, balanced=True, patience=4,
    ),
    "exp005_xlsr_dualgraph_lapa": dict(
        model="xlsr_dualgraph", tracks=("LA", "PA"), batch_size=2, eval_batch_size=4,
        grad_accum=8, epochs=12, lr=1e-4, backbone_lr=8e-7, weight_decay=1e-4,
        clip_samples=96_000, freeze=True, freeze_epochs=2, unfreeze_last_n=4,
        dropout=0.20, balanced=True, patience=5,
    ),
    "exp006_xlsr_dualgraph_rawboost": dict(
        model="xlsr_dualgraph", tracks=("LA", "PA"), batch_size=2, eval_batch_size=4,
        grad_accum=8, epochs=14, lr=8e-5, backbone_lr=5e-7, weight_decay=1e-4,
        clip_samples=96_000, freeze=True, freeze_epochs=3, unfreeze_last_n=4,
        rawboost_probability=0.35, rawboost_algo=5, communication_probability=0.25,
        dropout=0.25, balanced=True, patience=5,
    ),
    "exp007_xlsr_bimamba": dict(
        model="xlsr_mamba", tracks=("LA", "PA"), batch_size=1, eval_batch_size=2,
        grad_accum=16, epochs=12, lr=8e-5, backbone_lr=5e-7, weight_decay=1e-4,
        clip_samples=96_000, freeze=True, freeze_epochs=3, unfreeze_last_n=4,
        dropout=0.20, balanced=True, patience=5,
    ),
    # 공개 train의 multi-label 전용. target_cols/num_outputs 확정 후 사용.
    "exp008_speech_music_multibranch": dict(
        model="multibranch", tracks=(), batch_size=1, eval_batch_size=2,
        grad_accum=16, epochs=12, lr=1e-4, backbone_lr=5e-7, weight_decay=1e-4,
        clip_samples=96_000, freeze=True, freeze_epochs=3, unfreeze_last_n=4,
        dropout=0.25, balanced=False, patience=5, num_outputs=None,
    ),
}


def build_model(exp: dict) -> nn.Module:
    kind = exp["model"]
    if kind == "logmel":
        return LogMelCNN(dropout=exp.get("dropout", 0.25))
    if kind == "aasist":
        return load_official_aasist(freq_aug=exp.get("freq_aug", False))
    if kind == "xlsr_pool":
        return XLSRPoolClassifier(XLSR_MODEL, freeze=exp.get("freeze", True), dropout=exp.get("dropout", 0.2))
    if kind == "xlsr_dualgraph":
        return XLSRDualGraphClassifier(XLSR_MODEL, freeze=exp.get("freeze", True), dropout=exp.get("dropout", 0.2))
    if kind == "xlsr_mamba":
        return XLSRBiMambaClassifier(XLSR_MODEL, freeze=exp.get("freeze", True), dropout=exp.get("dropout", 0.2))
    if kind == "multibranch":
        if exp.get("num_outputs") is None:
            raise ValueError("대회 target_cols 공개 후 num_outputs를 지정하세요.")
        return SpeechMusicMultiBranch(XLSR_MODEL, exp["num_outputs"], freeze=exp.get("freeze", True), dropout=exp.get("dropout", 0.25))
    raise KeyError(kind)


def load_best_model(experiment_name: str, device: torch.device = DEVICE):
    checkpoint_path = RUN_ROOT / experiment_name / "best.pt"
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    saved_exp = dict(checkpoint["config"])
    loaded_model = build_model(saved_exp)
    loaded_model.load_state_dict(checkpoint["model_state"], strict=True)
    loaded_model.to(device).eval()
    print("loaded:", checkpoint_path, checkpoint.get("dev_metrics"))
    return loaded_model, saved_exp, checkpoint


## 14. 공통 Trainer — AMP, accumulation, early stopping, resume, dev prediction

체크포인트 선택은 binary ASV 실험에서 dev EER 최소값을 사용합니다. multi-label 대회 train 실험은 macro F1 최대값을 기본으로 하되, 공식 평가식 공개 후 바꿉니다.


In [ ]:
def class_weights_from_frame(frame: pd.DataFrame) -> torch.Tensor:
    counts = frame["label"].value_counts().reindex([0, 1], fill_value=1).to_numpy(dtype=np.float64)
    weights = np.sqrt(counts.sum() / (2 * counts))
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


def make_optimizer(model: nn.Module, exp: dict) -> torch.optim.Optimizer:
    backbone, head = [], []
    for name, parameter in model.named_parameters():
        (backbone if (name.startswith("ssl.") or name.startswith("net.ssl.")) else head).append(parameter)
    groups = []
    if backbone:
        groups.append({"params": backbone, "lr": exp.get("backbone_lr", exp["lr"])})
    if head:
        groups.append({"params": head, "lr": exp["lr"]})
    return torch.optim.AdamW(groups, weight_decay=exp["weight_decay"])


def probabilities_from_logits(logits: torch.Tensor, multilabel: bool = False) -> torch.Tensor:
    if logits.ndim != 2:
        raise ValueError(f"logits shape 오류: {tuple(logits.shape)}")
    if multilabel:
        return torch.sigmoid(logits)
    if logits.shape[1] != 2:
        raise ValueError(f"binary 모델은 logits 2개가 필요합니다: {tuple(logits.shape)}")
    return torch.softmax(logits, dim=-1)[:, 1]


def run_epoch(model, loader, criterion, optimizer=None, grad_accum=1, scaler=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    all_labels, all_probs, all_ids = [], [], []
    if training:
        optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(tqdm(loader, leave=False, desc="train" if training else "valid"), 1):
        audio = batch["audio"].to(DEVICE, non_blocking=True)
        label = batch["label"].to(DEVICE, non_blocking=True)
        amp_enabled = DEVICE.type == "cuda"
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=amp_enabled):
            logits = model(audio)
            loss = criterion(logits, label) / (grad_accum if training else 1)

        if training:
            scaler.scale(loss).backward()
            if step % grad_accum == 0 or step == len(loader):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

        total_loss += float(loss.detach().cpu()) * (grad_accum if training else 1)
        probabilities = probabilities_from_logits(logits.detach(), multilabel=label.ndim > 1).cpu().numpy()
        all_probs.append(probabilities)
        all_labels.append(label.detach().cpu().numpy())
        all_ids.extend(batch["id"])

    y_true = np.concatenate(all_labels)
    probabilities = np.concatenate(all_probs)
    metrics = binary_metrics(y_true, probabilities) if y_true.ndim == 1 else multilabel_metrics(y_true, probabilities)
    metrics["loss"] = total_loss / max(1, len(loader))
    return metrics, y_true, probabilities, all_ids


def fit_model(model, train_loader, dev_loader, train_frame, exp_name: str, exp: dict):
    run_dir = RUN_ROOT / exp_name
    run_dir.mkdir(parents=True, exist_ok=True)
    model = model.to(DEVICE)
    label_cols = exp.get("label_cols", "label")
    multilabel = not isinstance(label_cols, str) and len(label_cols) > 1
    if multilabel:
        positives = train_frame[list(label_cols)].sum(axis=0).to_numpy(dtype=np.float64)
        negatives = len(train_frame) - positives
        pos_weight = torch.tensor(negatives / np.clip(positives, 1, None), dtype=torch.float32, device=DEVICE)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        weights = None if exp.get("balanced", True) else class_weights_from_frame(train_frame)
        criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = make_optimizer(model, exp)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=exp["epochs"], eta_min=1e-7)
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")
    history, best, stale = [], (-float("inf") if multilabel else float("inf")), 0

    for epoch in range(1, exp["epochs"] + 1):
        if epoch == exp.get("freeze_epochs", -1) + 1 and hasattr(model, "unfreeze_last_n"):
            model.unfreeze_last_n(exp.get("unfreeze_last_n", 4))
            print("unfroze last SSL layers")
        train_metrics, *_ = run_epoch(
            model, train_loader, criterion, optimizer,
            grad_accum=exp.get("grad_accum", 1), scaler=scaler,
        )
        with torch.no_grad():
            dev_metrics, y_dev, p_dev, ids_dev = run_epoch(model, dev_loader, criterion)
        scheduler.step()
        record = {"epoch": epoch, **{f"train_{k}": v for k, v in train_metrics.items()}, **{f"dev_{k}": v for k, v in dev_metrics.items()}}
        history.append(record)
        print(json.dumps(record, ensure_ascii=False))
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)

        score = dev_metrics["macro_f1"] if multilabel else dev_metrics["eer"]
        improved = score > best if multilabel else score < best
        if improved:
            best, stale = score, 0
            torch.save({
                "model_state": model.state_dict(), "experiment": exp_name,
                "config": exp, "project_config": asdict(CFG), "repo_commits": REPO_COMMITS,
                "dev_metrics": dev_metrics, "label_convention": {"0": "spoof", "1": "bonafide"},
            }, run_dir / "best.pt")
            if multilabel:
                pred_frame = pd.DataFrame({"id": ids_dev})
                for idx, column in enumerate(label_cols):
                    pred_frame[f"label_{column}"] = y_dev[:, idx]
                    pred_frame[f"score_{column}"] = p_dev[:, idx]
            else:
                pred_frame = pd.DataFrame({"id": ids_dev, "label": y_dev, "score_bonafide": p_dev})
            pred_frame.to_csv(run_dir / "dev_predictions.csv", index=False)
        else:
            stale += 1
        if stale >= exp["patience"]:
            print("early stopping")
            break
    return pd.DataFrame(history), run_dir / "best.pt"


## 15. 실험 선택 및 학습

처음에는 `exp001_logmel_lapa`를 1 epoch로 바꿔 end-to-end smoke test한 뒤 정상 종료되면 원래 epoch로 돌립니다. 모델별로 런타임을 새로 시작하면 GPU 메모리 파편화를 줄일 수 있습니다.


In [ ]:
SELECTED_EXPERIMENT = "exp001_logmel_lapa"
RUN_TRAINING = False  # 데이터 경로와 shape 확인 후 True

exp = dict(EXPERIMENTS[SELECTED_EXPERIMENT])
print(json.dumps(exp, indent=2, ensure_ascii=False))

if RUN_TRAINING:
    if exp["model"] == "multibranch":
        schema = infer_schema(dacon_contract, DACON_SCHEMA)
        train_frame, dev_frame, target_cols, label_cols = prepare_dacon_supervised_frames(dacon_contract, schema)
        exp["label_cols"] = label_cols
        exp["num_outputs"] = 2 if len(target_cols) == 1 else len(target_cols)
    else:
        if asv_df.empty:
            raise RuntimeError("ASVspoof index가 비었습니다. ZIP 경로/압축 구조를 확인하세요.")
        selected = asv_df[asv_df["track"].isin(exp["tracks"])].copy()
        train_frame = selected[selected["split"] == "train"].reset_index(drop=True)
        dev_frame = selected[selected["split"] == "dev"].reset_index(drop=True)
        assert len(train_frame) and len(dev_frame)
        assert set(train_frame["id"]).isdisjoint(set(dev_frame["id"])), "train/dev leakage"

    train_loader, dev_loader = make_loaders(train_frame, dev_frame, exp)
    sanity = next(iter(train_loader))
    print("audio:", sanity["audio"].shape, "label:", sanity["label"].shape)
    model = build_model(exp)
    with torch.no_grad():
        test_logits = model(sanity["audio"][:2].to(DEVICE))
    print("logits:", test_logits.shape)
    expected_outputs = exp.get("num_outputs") or 2
    assert test_logits.shape == (min(2, len(sanity["audio"])), expected_outputs)

    history, best_path = fit_model(model, train_loader, dev_loader, train_frame, SELECTED_EXPERIMENT, exp)
    print("best checkpoint:", best_path)


## 16. ASVspoof 2019 공식 eval 평가

하이퍼파라미터와 ensemble 가중치는 dev/OOF에서만 결정합니다. 설정을 고정한 뒤 공식 eval split은 최종 비교용으로 한 번 평가합니다. LA/PA별 EER을 따로 기록해 혼합 평균이 한 domain의 실패를 가리지 않게 합니다.


In [ ]:
@torch.no_grad()
def evaluate_asvspoof_eval(model: nn.Module, frame: pd.DataFrame, exp: dict, experiment_name: str):
    output_rows = []
    metrics_by_track = {}
    criterion = nn.CrossEntropyLoss()
    for track in exp["tracks"]:
        eval_frame = frame[(frame["split"] == "eval") & (frame["track"] == track)].reset_index(drop=True)
        if eval_frame.empty:
            print(f"[skip] {track} eval 없음")
            continue
        dataset = AudioFrameDataset(eval_frame, train=False, clip_samples=exp["clip_samples"])
        loader = DataLoader(
            dataset, batch_size=exp.get("eval_batch_size", exp["batch_size"]),
            shuffle=False, num_workers=CFG.num_workers, pin_memory=True,
        )
        metrics, labels, probabilities, identifiers = run_epoch(model, loader, criterion)
        metrics_by_track[track] = metrics
        output_rows.append(pd.DataFrame({
            "id": identifiers, "track": track, "label": labels,
            "score_bonafide": probabilities,
        }))
        print(track, metrics)
    if output_rows:
        output = pd.concat(output_rows, ignore_index=True)
        run_dir = RUN_ROOT / experiment_name
        output.to_csv(run_dir / "eval_predictions.csv", index=False)
        (run_dir / "eval_metrics.json").write_text(
            json.dumps(metrics_by_track, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return metrics_by_track, output
    return {}, pd.DataFrame()


RUN_ASVSPOOF_EVAL = False
if RUN_ASVSPOOF_EVAL:
    model, exp, checkpoint = load_best_model(SELECTED_EXPERIMENT)
    eval_metrics, eval_predictions = evaluate_asvspoof_eval(
        model, asv_df, exp, SELECTED_EXPERIMENT
    )


## 17. 하이퍼파라미터 탐색 원칙

TEST가 아니라 dev/OOF에서만 탐색합니다. 아래 search space를 기준으로 2–3 epoch의 저비용 trial을 거친 뒤, 상위 2–3개 설정만 full epoch로 재학습하세요. 서로 다른 seed까지 비교하지 않으면 우연한 차이를 “최적”으로 오인하기 쉽습니다.

권장 탐색 범위:

| 모델 | lr(head) | lr(backbone) | weight decay | 기타 |
|---|---:|---:|---:|---|
| LogMel | 1e-4–1e-3 | 동일 | 1e-6–1e-3 | n_mels 80/96/128, dropout .1–.4 |
| AASIST | 3e-5–3e-4 | 동일 | 1e-6–5e-4 | RawBoost p 0/.25/.5, algo 5/8 |
| XLS-R | 5e-5–3e-4 | 1e-7–3e-6 | 1e-6–5e-4 | freeze 1–4 epochs, last 2/4/6 layers |
| Mamba | 3e-5–2e-4 | 1e-7–1e-6 | 1e-6–5e-4 | d_model 128/256, d_state 8/16 |


In [ ]:
import optuna

RUN_OPTUNA = False
OPTUNA_BASE_EXPERIMENT = "exp001_logmel_lapa"


def optuna_objective(trial: optuna.Trial) -> float:
    base = dict(EXPERIMENTS[OPTUNA_BASE_EXPERIMENT])
    base.update({
        "lr": trial.suggest_float("lr", 1e-4, 1e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "dropout": trial.suggest_float("dropout", 0.1, 0.4),
        "epochs": 3,
        "patience": 3,
    })
    selected = asv_df[asv_df["track"].isin(base["tracks"])]
    train_frame = selected[selected["split"] == "train"].sample(frac=0.25, random_state=CFG.seed)
    dev_frame = selected[selected["split"] == "dev"].sample(frac=0.5, random_state=CFG.seed)
    train_loader, dev_loader = make_loaders(train_frame, dev_frame, base)
    model = build_model(base)
    history, _ = fit_model(model, train_loader, dev_loader, train_frame, f"optuna_trial_{trial.number:03d}", base)
    value = float(history["dev_eer"].min())
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return value


if RUN_OPTUNA:
    study = optuna.create_study(direction="minimize", study_name="deepvoice_dev_eer")
    study.optimize(optuna_objective, n_trials=20)
    print(study.best_value, study.best_params)


## 18. 공식 SSL_Anti-spoofing / XLSR-Mamba 논문 재현(별도 Colab 런타임)

두 공식 코드는 pinned legacy fairseq와 오래된 PyTorch 환경을 전제로 합니다. 현재 transformers 파이프라인과 같은 런타임에 강제로 섞으면 의존성 충돌 가능성이 큽니다.

1. `CLONE_LEGACY_ADVANCED_REPOS=True`로 clone하고 commit을 기록합니다.
2. 새 Colab 런타임에서 각 저장소 README의 fairseq commit과 requirements를 설치합니다.
3. ASVspoof 2019 LA train/dev 경로로 먼저 공식 command를 재현합니다.
4. 공식 checkpoint와 dev score 파일을 `RUN_ROOT/exp007_official_*`에 복사합니다.
5. 아래 ensemble 형식(`id,label,score_bonafide`)으로 score를 변환합니다.

공식 SSL 구현의 공개 시작값은 LA에서 `lr=1e-6`, `batch_size=14`, weighted CE입니다. T4 메모리가 부족하면 batch 2와 accumulation 7을 사용합니다. 공식 XLSR-Mamba는 fixed-length input의 `--algo 5`를 제시합니다. 구현·가중치 출처와 license는 반드시 2차 보고서에 기록하세요.


## 19. Dev/OOF 기반 ensemble, calibration, threshold

가중치는 임의로 정하지 않고 dev/OOF에서 결정합니다. TEST 예측끼리 통계적으로 보정하지 않습니다. 공식 제출이 확률을 요구하면 threshold를 적용하지 않고 calibration된 확률을 제출합니다.


In [ ]:
def load_prediction_files(paths: Sequence[str | Path]) -> tuple[pd.DataFrame, np.ndarray]:
    frames = [pd.read_csv(path).rename(columns={"score_bonafide": f"m{i}"}) for i, path in enumerate(paths)]
    merged = frames[0]
    for i, frame in enumerate(frames[1:], 1):
        merged = merged.merge(frame[["id", f"m{i}"]], on="id", validate="one_to_one")
    model_cols = [c for c in merged if c.startswith("m")]
    return merged, merged[model_cols].to_numpy()


def optimize_weights(y_true: np.ndarray, predictions: np.ndarray) -> np.ndarray:
    n_models = predictions.shape[1]
    def objective(weights):
        score = np.clip(predictions @ weights, 1e-6, 1 - 1e-6)
        return log_loss(y_true, score)
    result = minimize(
        objective, np.full(n_models, 1 / n_models), method="SLSQP",
        bounds=[(0, 1)] * n_models,
        constraints={"type": "eq", "fun": lambda w: w.sum() - 1},
    )
    if not result.success:
        raise RuntimeError(result.message)
    return result.x


def optimize_multilabel_weights(y_true: np.ndarray, predictions: np.ndarray) -> np.ndarray:
    """predictions: [N, M, C]. 각 target C마다 OOF 기반 model weight를 학습."""
    if predictions.ndim != 3 or y_true.shape != (predictions.shape[0], predictions.shape[2]):
        raise ValueError((y_true.shape, predictions.shape))
    return np.stack([
        optimize_weights(y_true[:, target], predictions[:, :, target])
        for target in range(y_true.shape[1])
    ])


def apply_multilabel_weights(predictions: np.ndarray, weights: np.ndarray) -> np.ndarray:
    return np.einsum("nmc,cm->nc", predictions, weights)


def fit_platt(y_true: np.ndarray, scores: np.ndarray) -> LogisticRegression:
    logits = np.log(np.clip(scores, 1e-6, 1 - 1e-6) / np.clip(1 - scores, 1e-6, 1))
    return LogisticRegression(C=1.0).fit(logits.reshape(-1, 1), y_true)


def optimize_f1_threshold(y_true: np.ndarray, scores: np.ndarray) -> tuple[float, float]:
    candidates = np.linspace(0.02, 0.98, 193)
    values = np.array([f1_score(y_true, scores >= threshold, zero_division=0) for threshold in candidates])
    index = int(values.argmax())
    return float(candidates[index]), float(values[index])


DEV_PREDICTION_FILES = []  # 각 실험의 RUN_ROOT/.../dev_predictions.csv
if DEV_PREDICTION_FILES:
    merged_dev, dev_matrix = load_prediction_files(DEV_PREDICTION_FILES)
    ensemble_weights = optimize_weights(merged_dev["label"].to_numpy(), dev_matrix)
    raw_ensemble = dev_matrix @ ensemble_weights
    calibrator = fit_platt(merged_dev["label"].to_numpy(), raw_ensemble)
    raw_logit = np.log(np.clip(raw_ensemble, 1e-6, 1 - 1e-6) / np.clip(1 - raw_ensemble, 1e-6, 1))
    calibrated = calibrator.predict_proba(raw_logit.reshape(-1, 1))[:, 1]
    print("weights:", ensemble_weights)
    print("raw:", binary_metrics(merged_dev["label"].to_numpy(), raw_ensemble))
    print("calibrated:", binary_metrics(merged_dev["label"].to_numpy(), calibrated))


## 20. DACON TEST 독립 추론

각 오디오 안에서 최대 5개 deterministic segment를 평균합니다. 이것은 공식 규칙상 허용된 file-internal segment inference입니다. `sample_submission.csv`의 행 순서를 보존하고, 단일 target일 때 `single_target_positive` 방향을 명시적으로 적용합니다.


In [ ]:
def infer_schema(contract: dict, overrides: dict) -> dict:
    sample = contract.get("sample")
    test = contract.get("test")
    if sample is None:
        raise FileNotFoundError("sample_submission.csv가 없습니다. 8월 26일 공개 데이터를 확인하세요.")
    id_col = overrides.get("id_col") or sample.columns[0]
    target_cols = overrides.get("target_cols") or [c for c in sample.columns if c != id_col]
    audio_col = overrides.get("audio_col")
    if audio_col is None and test is not None:
        candidates = [c for c in test.columns if any(k in c.lower() for k in ("path", "file", "audio", "name"))]
        audio_col = candidates[0] if candidates else None
    return {**overrides, "id_col": id_col, "target_cols": target_cols, "audio_col": audio_col}


def resolve_dacon_test_frame(contract: dict, schema: dict) -> pd.DataFrame:
    sample = contract["sample"].copy()
    test = contract.get("test")
    audio = contract["audio"].copy()
    id_col = schema["id_col"]
    sample["_join_key"] = sample[id_col].astype(str).map(lambda x: Path(x).stem)
    if test is not None and schema["audio_col"] is not None:
        test = test.copy()
        refs = test[schema["audio_col"]].astype(str)
        test["id"] = test[id_col].astype(str) if id_col in test else refs
        test["_join_key"] = test["id"].map(lambda x: Path(x).stem)
        by_stem = dict(zip(audio["id"].astype(str), audio["path"]))
        test["path"] = refs.map(lambda x: str(DACON_ROOT / x) if (DACON_ROOT / x).exists() else by_stem.get(Path(x).stem, ""))
        frame = sample[[id_col, "_join_key"]].merge(test[["id", "_join_key", "path"]], on="_join_key", how="left")
    else:
        audio["_join_key"] = audio["id"].astype(str).map(lambda x: Path(x).stem)
        frame = sample[[id_col, "_join_key"]].merge(audio, on="_join_key", how="left")
    if frame["path"].isna().any() or (frame["path"] == "").any():
        raise FileNotFoundError("sample_submission 행과 TEST 오디오를 모두 매칭하지 못했습니다.")
    return frame


def deterministic_segments(wav: torch.Tensor, length: int, count: int = 5) -> torch.Tensor:
    if wav.numel() <= length:
        return fixed_length(wav, length, train=False).unsqueeze(0)
    starts = np.linspace(0, wav.numel() - length, num=count, dtype=int)
    return torch.stack([wav[start : start + length] for start in starts])


@torch.no_grad()
def predict_files(model: nn.Module, frame: pd.DataFrame, clip_samples: int, segments: int = 5, multilabel: bool = False) -> np.ndarray:
    model.eval().to(DEVICE)
    output = []
    for row in tqdm(frame.itertuples(index=False), total=len(frame), desc="DACON inference"):
        wav = load_mono(row.path, CFG.sample_rate)
        clips = deterministic_segments(wav, clip_samples, segments)
        logits = model(clips.to(DEVICE))
        probs = probabilities_from_logits(logits, multilabel=multilabel).mean(dim=0).cpu().numpy()
        output.append(np.atleast_1d(probs))
    return np.stack(output)


def make_submission(contract: dict, schema: dict, probabilities: np.ndarray, output_path: Path) -> pd.DataFrame:
    sample = contract["sample"].copy()
    targets = schema["target_cols"]
    if probabilities.shape[1] == 1 and len(targets) == 1:
        p_bonafide = probabilities[:, 0]
        values = 1 - p_bonafide if schema["single_target_positive"] == "spoof" else p_bonafide
        sample[targets[0]] = values
    elif probabilities.shape[1] == len(targets):
        sample.loc[:, targets] = probabilities
    else:
        raise ValueError(f"model outputs={probabilities.shape[1]}, submission targets={len(targets)}")
    if sample[targets].isna().any().any():
        raise ValueError("submission에 NaN이 있습니다.")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    sample.to_csv(output_path, index=False, encoding="utf-8")
    return sample


In [ ]:
RUN_DACON_INFERENCE = False

if RUN_DACON_INFERENCE:
    schema = infer_schema(dacon_contract, DACON_SCHEMA)
    print(schema)
    test_frame = resolve_dacon_test_frame(dacon_contract, schema)
    # model은 15절에서 학습했거나 best.pt를 build_model(exp)에 load한 객체여야 합니다.
    test_probabilities = predict_files(
        model, test_frame, exp["clip_samples"], segments=5,
        multilabel=len(schema["target_cols"]) > 1,
    )
    submission = make_submission(
        dacon_contract, schema, test_probabilities,
        EXPORT_ROOT / "local_submission.csv",
    )
    display(submission.head())


## 21. 코드 제출용 `submit.zip` 생성

아래 exporter는 최종 모델을 fixed-length TorchScript로 저장하고, 오프라인 `script.py`가 `/data`(또는 ZIP 옆 `data`)를 읽어 `output/submission.csv`를 생성하게 합니다. 대회 공개 후 반드시 서버 기본 패키지·용량·제한 시간에 맞춰 smoke test하세요. XLS-R 300M ensemble은 ZIP 제한을 초과할 수 있으므로 distillation 또는 모델 수 축소가 필요할 수 있습니다.


In [ ]:
SUBMISSION_SCRIPT = r'''
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from scipy.signal import resample_poly

BASE = Path(__file__).resolve().parent
DATA = BASE / "data" if (BASE / "data").exists() else Path("/data")
OUTPUT = BASE / "output"
OUTPUT.mkdir(parents=True, exist_ok=True)
META = json.loads((BASE / "model" / "metadata.json").read_text(encoding="utf-8"))
MODEL = torch.jit.load(str(BASE / "model" / "model.ts"), map_location="cpu").eval()
SUFFIXES = {".wav", ".flac", ".ogg", ".mp3", ".m4a", ".aac"}

def load_mono(path):
    x, sr = sf.read(path, dtype="float32", always_2d=True)
    x = x.mean(axis=1)
    if sr != META["sample_rate"]:
        divisor = math.gcd(sr, META["sample_rate"])
        x = resample_poly(x, META["sample_rate"] // divisor, sr // divisor).astype("float32")
    if len(x) == 0:
        raise ValueError(f"empty audio: {path}")
    return torch.from_numpy(x)

def segments(x):
    length = META["clip_samples"]
    if len(x) <= length:
        x = x.repeat(math.ceil(length / len(x)))[:length]
        return x.unsqueeze(0)
    starts = np.linspace(0, len(x) - length, META["segments"], dtype=int)
    return torch.stack([x[s:s+length] for s in starts])

sample_paths = sorted(DATA.rglob("*sample*submission*.csv"))
if len(sample_paths) != 1:
    raise FileNotFoundError(f"sample_submission count={len(sample_paths)}")
sample = pd.read_csv(sample_paths[0])
id_col = META["id_col"]
targets = META["target_cols"]
audio = {p.stem: p for p in DATA.rglob("*") if p.is_file() and p.suffix.lower() in SUFFIXES}
predictions = []
with torch.inference_mode():
    for identifier in sample[id_col].astype(str):
        audio_key = Path(identifier).stem
        if audio_key not in audio:
            raise FileNotFoundError(f"audio not found: {identifier}")
        predictions.append(MODEL(segments(load_mono(audio[audio_key]))).mean(0).numpy())
predictions = np.stack(predictions)
if predictions.ndim == 1:
    predictions = predictions[:, None]
if predictions.shape[1] != len(targets):
    raise ValueError((predictions.shape, targets))
sample.loc[:, targets] = predictions
if sample[targets].isna().any().any():
    raise ValueError("NaN in submission")
sample.to_csv(OUTPUT / "submission.csv", index=False, encoding="utf-8")
'''


class SubmissionProbabilityWrapper(nn.Module):
    def __init__(self, base_model: nn.Module, num_targets: int, single_target_positive: str = "spoof"):
        super().__init__()
        self.base_model = base_model
        self.num_targets = num_targets
        self.spoof_positive = single_target_positive == "spoof"

    def forward(self, audio: torch.Tensor) -> torch.Tensor:
        logits = self.base_model(audio)
        if self.num_targets == 1:
            if logits.shape[-1] != 2:
                raise RuntimeError("single target binary 모델은 logits 2개가 필요합니다.")
            p_bonafide = torch.softmax(logits, dim=-1)[:, 1:2]
            return 1 - p_bonafide if self.spoof_positive else p_bonafide
        return torch.sigmoid(logits)


def build_submit_zip(model: nn.Module, schema: dict, exp: dict, destination: Path) -> Path:
    package = WORK_ROOT / "submit_package"
    if package.exists():
        shutil.rmtree(package)
    model_dir = package / "model"
    model_dir.mkdir(parents=True)
    wrapper = SubmissionProbabilityWrapper(
        model.eval().cpu(), len(schema["target_cols"]), schema["single_target_positive"]
    )
    example = torch.zeros(1, exp["clip_samples"])
    traced = torch.jit.trace(wrapper, example, strict=False)
    traced.save(str(model_dir / "model.ts"))
    metadata = {
        "sample_rate": CFG.sample_rate, "clip_samples": exp["clip_samples"],
        "segments": 5, "id_col": schema["id_col"], "target_cols": schema["target_cols"],
        "single_target_positive": schema["single_target_positive"],
        "experiment": SELECTED_EXPERIMENT, "repo_commits": REPO_COMMITS,
    }
    (model_dir / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    (package / "script.py").write_text(SUBMISSION_SCRIPT, encoding="utf-8")
    (package / "requirements.txt").write_text("soundfile\nscipy\npandas\n", encoding="utf-8")

    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        destination.unlink()
    shutil.make_archive(str(destination.with_suffix("")), "zip", package)
    with zipfile.ZipFile(destination) as zf:
        names = set(zf.namelist())
        assert "script.py" in names and "requirements.txt" in names
        assert "model/model.ts" in names and "model/metadata.json" in names
    print("created:", destination, "MB=", destination.stat().st_size / 1024**2)
    return destination


BUILD_SUBMIT_ZIP = False
if BUILD_SUBMIT_ZIP:
    schema = infer_schema(dacon_contract, DACON_SCHEMA)
    submit_zip = build_submit_zip(model, schema, exp, EXPORT_ROOT / "submit.zip")


## 22. 제출 전 필수 smoke test

1. 대회 공개 후 평가식·target 의미·sample submission 열을 `DACON_SCHEMA`에 확정한다.
2. TEST는 어떤 학습·튜닝·pseudo-label에도 사용하지 않는다.
3. best checkpoint를 새 런타임에서 load해 dev 점수가 재현되는지 확인한다.
4. `submit.zip`을 임시 폴더에 풀고 인터넷을 사용하지 않은 상태에서 `python script.py`를 실행한다.
5. 결과가 정확히 `output/submission.csv`, UTF-8, sample과 동일한 행 순서/열 순서/행 수인지 확인한다.
6. 각 TEST 파일 prediction이 다른 TEST 파일의 통계나 prediction에 의존하지 않는지 확인한다.
7. ZIP 루트에 추가 상위 폴더가 없는지, 용량·설치·추론 제한을 충족하는지 확인한다.
8. 사용한 ASVspoof 파일, pretrained weight, 저장소 URL·commit·license를 학습데이터 구성 보고서에 기록한다.

### 권장 실제 순서

`EXP01 smoke → EXP02 LA → EXP02P PA → EXP03 mixed RawBoost → EXP04 SSL pooling → EXP05/06 dual graph → EXP07 Mamba → (공개 label 확인 후) EXP08 multi-branch → OOF ensemble/calibration → offline submit.zip smoke test`

단계별 결과는 `MyDrive/deepvoice_dacon/runs/<experiment>/history.csv`, `best.pt`, `dev_predictions.csv`에 남습니다.
